In [34]:
import os
os.environ["GEMINI_API_KEY"] = "AI..."

In [35]:
!pip install langchain chromadb google-generativeai tiktoken pypdf langchain-google-genai langchain-community

In [36]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [37]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [38]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [53]:
import chromadb
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Set up Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")


# Create the LangChain Vector Store using the clean client
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='chroma_db',
    collection_name='sample'
)

In [48]:
# add documents
vector_store.add_documents(docs)

['a03fa139-a735-40ac-bbe9-84e7ca8d082c',
 'e1951aaa-386c-4064-9e0e-d8cf683d0950',
 '4e4c8bce-de82-4d85-b2c2-65804acaa642',
 'd56d9d01-6435-4a61-a756-f863ad1effdb',
 '13b00171-9402-4839-abeb-200509f180bc']

In [49]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['a03fa139-a735-40ac-bbe9-84e7ca8d082c',
  'e1951aaa-386c-4064-9e0e-d8cf683d0950',
  '4e4c8bce-de82-4d85-b2c2-65804acaa642',
  'd56d9d01-6435-4a61-a756-f863ad1effdb',
  '13b00171-9402-4839-abeb-200509f180bc'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [50]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [51]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6193535327911377),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6677330136299133)]

In [54]:
# Option 1: Semantic search with metadata filtering (requires a non-empty query)
results = vector_store.similarity_search_with_score(
    query="Captain Cool",
    filter={"team": "Chennai Super Kings"}
)
print("Search Results:", results)

# Option 2: Pure metadata filtering (no query or embeddings required)
# filtered_docs = vector_store.get(where={"team": "Chennai Super Kings"})
# print("Filtered Docs:", filtered_docs)

Search Results: [(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'), 0.7725858092308044), (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'), 0.986804723739624)]


In [55]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='a03fa139-a735-40ac-bbe9-84e7ca8d082c', document=updated_doc1)


In [56]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['a03fa139-a735-40ac-bbe9-84e7ca8d082c',
  'e1951aaa-386c-4064-9e0e-d8cf683d0950',
  '4e4c8bce-de82-4d85-b2c2-65804acaa642',
  'd56d9d01-6435-4a61-a756-f863ad1effdb',
  '13b00171-9402-4839-abeb-200509f180bc'],
 'embeddings': array([[ 0.01820879,  0.0189696 ,  0.00767588, ..., -0.00233459,
          0.0060162 ,  0.01336217],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a sin

In [57]:
# delete document
vector_store.delete(ids=['a03fa139-a735-40ac-bbe9-84e7ca8d082c'])

In [58]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['e1951aaa-386c-4064-9e0e-d8cf683d0950',
  '4e4c8bce-de82-4d85-b2c2-65804acaa642',
  'd56d9d01-6435-4a61-a756-f863ad1effdb',
  '13b00171-9402-4839-abeb-200509f180bc'],
 'embeddings': array([[ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]]),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one 